In [32]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix

In [33]:
movies = pd.read_csv("../data/processed_movies.csv")
ratings = pd.read_csv("../data/ratings.csv")

movies['metadata'] = movies['metadata'].fillna('').astype(str)
display(movies.head(3))
display(ratings.head(3))

,movieId,title,genres,clean_title,tag,metadata
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,Toy Story,children Disney animation children Disney Disn...,Toy Story Adventure Animation Children Comedy...
1,2,Jumanji (1995),Adventure Children Fantasy,Jumanji,Robin Williams fantasy Robin Williams time tra...,Jumanji Adventure Children Fantasy Robin Will...
2,3,Grumpier Old Men (1995),Comedy Romance,Grumpier Old Men,comedinha de velhinhos engraÃƒÂ§ada comedinha ...,Grumpier Old Men Comedy Romance comedinha de ...


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976


<H>CONTENT BASED</H>

In [34]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['metadata'])

#cos_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

In [35]:
def content_recommend(title, num=10, round=4):

    matches = movies[
        movies['title']
        .str.lower()
        .str.contains(title.lower(), regex=False)
    ]

    if matches.empty:
        return "Movie not found."

    title = matches.iloc[0]['title']
    #print(f"Recommending for: {title}")
        
    idx = indices[title]

    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    sim_scores = list(enumerate(sim_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:num+1]

    movie_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]

    result = movies[['movieId', 'title', 'metadata']].iloc[movie_indices].copy()
    result['similarity_score'] = scores
    result['similarity_score'] = result['similarity_score'].round(round)

    result = result.reset_index(drop=True)

    return result

<H>COLLABORATIVE FILTERING</H>

In [36]:
movie_rating_counts = ratings.groupby('movieId').size()

movie_ids = np.sort(ratings['movieId'].unique())
user_ids = np.sort(ratings['userId'].unique())

movie_id_to_col = pd.Series(
    np.arange(len(movie_ids)),
    index=movie_ids
)

user_id_to_row = pd.Series(
    np.arange(len(user_ids)),
    index=user_ids
)

movie_col_to_id = pd.Series(
    movie_ids,
    index=np.arange(len(movie_ids))
)

user_row_to_id = pd.Series(
    user_ids,
    index=np.arange(len(user_ids))
)

In [37]:
user_rows = ratings['userId'].map(user_id_to_row).to_numpy()
movie_cols = ratings['movieId'].map(movie_id_to_col).to_numpy()
rating_values = ratings['rating'].astype('float32').to_numpy()

user_movie_sparse = csr_matrix(
    (
        rating_values,
        (user_rows, movie_cols)
    ),
    shape=(
        len(user_ids),
        len(movie_ids)
    ),
    dtype=np.float32
)

user_movie_norm = normalize(user_movie_sparse, axis=1)
user_mean_ratings = ratings.groupby('userId')['rating'].mean()

movie_user_sparse = user_movie_sparse.T.tocsr()
movie_user_norm = normalize(movie_user_sparse, axis=1)

In [38]:
normalized_values = (
    ratings['rating'].astype('float32')
    -
    ratings['userId'].map(user_mean_ratings).astype('float32')
).to_numpy()

normalized_user_movie_sparse = csr_matrix(
    (
        normalized_values,
        (user_rows, movie_cols)
    ),
    shape=(len(user_ids), len(movie_ids)),
    dtype=np.float32
)

normalized_user_movie_norm = normalize(normalized_user_movie_sparse, axis=1)

In [39]:
def get_normalized_similar_users(user_id, n=50):

    if user_id not in user_id_to_row.index:
        return "Invalid user ID"

    user_row = user_id_to_row[user_id]

    similarity_scores = (
        normalized_user_movie_norm[user_row]
        .dot(normalized_user_movie_norm.T)
        .toarray()
        .flatten()
    )
    similarity_scores[user_row] = -1

    sorted_indices = np.argsort(similarity_scores)[::-1]
    top_indices = sorted_indices[:n]

    similar_user_ids = user_row_to_id.iloc[top_indices].values
    similar_scores = similarity_scores[top_indices]
    similar_users = pd.Series(similar_scores, index=similar_user_ids)

    return similar_users

In [40]:
def find_movie(title):
    
    matches = movies[
        movies['title']
        .str.lower()
        .str.contains(title.lower(), regex=False)
    ]

    if matches.empty:
        return "Movie not found."
    
    return matches.iloc[0]

In [41]:
def collaborative_recommend(title, n_movies=10, min_ratings=100, round=4):
    
    movie = find_movie(title)

    if movie is None:
        return "Movie not found."
    
    title = movie['title']
    movie_id = movie['movieId']

    #print(f"Recommending for: {title}")

    if movie_id not in movie_id_to_col.index:
        return "This movie has no rating data"
    
    movie_col = movie_id_to_col[movie_id]

    similarity_scores = (
        movie_user_norm[movie_col]
        .dot(movie_user_norm.T)
        .toarray()
        .flatten()
    )
    similarity_scores[movie_col] = -1

    sorted_indices = np.argsort(similarity_scores)[::-1]
    candidate_movie_ids = movie_col_to_id.iloc[sorted_indices].values
    candidate_scores = similarity_scores[sorted_indices]

    similar_movies = pd.Series(candidate_scores, index=candidate_movie_ids)
    similar_movies = similar_movies[similar_movies > 0]
    similar_movies = similar_movies[
        movie_rating_counts.reindex(similar_movies.index).fillna(0) >= min_ratings
    ]
    similar_movies = similar_movies.head(n_movies)

    recommended_movies = movies[
        movies['movieId'].isin(similar_movies.index)
    ][['movieId', 'title', 'metadata']].copy()

    recommended_movies['similarity_score'] = (
        recommended_movies['movieId']
        .map(similar_movies)
        .round(round)
    )

    recommended_movies = (
        recommended_movies
        .sort_values(by='similarity_score', ascending=False)
        .reset_index(drop=True)
    )
    
    return recommended_movies


<H>HYBRID RECOMMENDATION</H>

In [42]:
def  get_content_weight(rating_count):

    if rating_count >= 1000:
        return 0.2
    elif rating_count >= 100:
        return 0.4
    else:
        return 0.7

In [43]:
def hybrid_recommend(title, n_movies=10, min_ratings=20, candidate_pool=100, round=4):
    
    movie = find_movie(title)

    if movie is None:
        return "Movie not found."
    
    title = movie['title']
    movie_id = movie['movieId']

    print(f"Recommending for: {title}")

    rating_count = movie_rating_counts.get(movie_id, 0)

    content_weight = get_content_weight(rating_count)
    collaborative_weight = 1.0 - content_weight

    content_df = content_recommend(title, num=candidate_pool, round=round)
    collaborative_df = collaborative_recommend(title, n_movies=candidate_pool, min_ratings=min_ratings, round=round)

    if isinstance(content_df, str):
        content_df = pd.DataFrame(
            columns=[
                'movieId',
                'title',
                'metadata',
                'similarity_score'
            ]
        )

    if isinstance(collaborative_df, str):
        collaborative_df = pd.DataFrame(
            columns=[
                'movieId',
                'title',
                'metadata',
                'similarity_score'
            ]
        )
    
    content_df = content_df.rename(columns={'similarity_score': 'content_score'})
    collaborative_df = collaborative_df.rename(columns={'similarity_score': 'collaborative_score'})

    hybrid_df = pd.merge(
        content_df, collaborative_df,
        on=['movieId', 'title'],
        how='outer'
    )

    hybrid_df['metadata'] = hybrid_df['metadata_x'].fillna(hybrid_df['metadata_y'])
    hybrid_df = hybrid_df.drop(columns=['metadata_x', 'metadata_y'])
    hybrid_df['content_score'] = hybrid_df['content_score'].fillna(0)
    hybrid_df['collaborative_score'] = hybrid_df['collaborative_score'].fillna(0)

    hybrid_df['hybrid_score'] = (
        hybrid_df['content_score'] * content_weight
        +
        hybrid_df['collaborative_score'] * collaborative_weight
    ).round(round)

    hybrid_df = (
        hybrid_df
        .sort_values(by='hybrid_score', ascending=False)
        .reset_index(drop=True)
    )

    hybrid_df = hybrid_df[
        [
            'movieId',
            'title',
            'metadata',
            'content_score',
            'collaborative_score',
            'hybrid_score'
        ]
    ]

    print(f"Rating count: {rating_count}")
    print(f"Content weight: {content_weight}")
    print(f"Collaborative weight: {collaborative_weight}")

    return hybrid_df.head(n_movies)
    

In [53]:
hybrid_recommend('Toy Story', n_movies=10, min_ratings=100, candidate_pool=500, round=3)

Recommending for: Toy Story (1995)
Rating count: 68997
Content weight: 0.2
Collaborative weight: 0.8


,movieId,title,metadata,content_score,collaborative_score,hybrid_score
0,3114,Toy Story 2 (1999),Toy Story 2 Adventure Animation Children Come...,0.888,0.575,0.638
1,4886,"Monsters, Inc. (2001)","Monsters, Inc. Adventure Animation Children C...",0.699,0.513,0.550
2,6377,Finding Nemo (2003),Finding Nemo Adventure Animation Children Com...,0.693,0.501,0.539
3,2355,"Bug's Life, A (1998)","Bug's Life, A Adventure Animation Children Co...",0.804,0.466,0.534
4,8961,"Incredibles, The (2004)","Incredibles, The Action Adventure Animation C...",0.683,0.474,0.516
5,356,Forrest Gump (1994),Forrest Gump Comedy Drama Romance War bitters...,0.250,0.542,0.484
6,364,"Lion King, The (1994)","Lion King, The Adventure Animation Children D...",0.246,0.537,0.479
7,4306,Shrek (2001),Shrek Adventure Animation Children Comedy Fan...,0.334,0.514,0.478
8,588,Aladdin (1992),Aladdin Adventure Animation Children Comedy M...,0.263,0.527,0.474
9,78499,Toy Story 3 (2010),Toy Story 3 Adventure Animation Children Come...,0.707,0.402,0.463


In [54]:
hybrid_recommend('Shrek', n_movies=10, min_ratings=100, candidate_pool=500, round=3)

Recommending for: Shrek (2001)
Rating count: 54844
Content weight: 0.2
Collaborative weight: 0.8


,movieId,title,metadata,content_score,collaborative_score,hybrid_score
0,8360,Shrek 2 (2004),Shrek 2 Adventure Animation Children Comedy M...,0.862,0.623,0.671
1,4886,"Monsters, Inc. (2001)","Monsters, Inc. Adventure Animation Children C...",0.305,0.679,0.604
2,6377,Finding Nemo (2003),Finding Nemo Adventure Animation Children Com...,0.308,0.664,0.593
3,8961,"Incredibles, The (2004)","Incredibles, The Action Adventure Animation C...",0.268,0.603,0.536
4,6539,Pirates of the Caribbean: The Curse of the Bla...,Pirates of the Caribbean: The Curse of the Bla...,0.000,0.629,0.503
5,4993,"Lord of the Rings: The Fellowship of the Ring,...","Lord of the Rings: The Fellowship of the Ring,...",0.000,0.625,0.500
6,5218,Ice Age (2002),Ice Age Adventure Animation Children Comedy a...,0.339,0.532,0.493
7,5952,"Lord of the Rings: The Two Towers, The (2002)","Lord of the Rings: The Two Towers, The Advent...",0.000,0.604,0.483
8,1,Toy Story (1995),Toy Story Adventure Animation Children Comedy...,0.334,0.514,0.478
9,5349,Spider-Man (2002),Spider-Man Action Adventure Sci-Fi Thriller d...,0.000,0.592,0.474


In [55]:
hybrid_recommend('Matrix', n_movies=10, min_ratings=100, candidate_pool=500, round=3)

Recommending for: Matrix, The (1999)
Rating count: 93808
Content weight: 0.2
Collaborative weight: 0.8


,movieId,title,metadata,content_score,collaborative_score,hybrid_score
0,2959,Fight Club (1999),Fight Club Action Crime Drama Thriller Brad P...,0.209,0.693,0.596
1,6365,"Matrix Reloaded, The (2003)","Matrix Reloaded, The Action Adventure Sci-Fi ...",0.859,0.527,0.593
2,79132,Inception (2010),Inception Action Crime Drama Mystery Sci-Fi T...,0.414,0.598,0.561
3,260,Star Wars: Episode IV - A New Hope (1977),Star Wars: Episode IV - A New Hope Action Adv...,0.203,0.640,0.553
4,4993,"Lord of the Rings: The Fellowship of the Ring,...","Lord of the Rings: The Fellowship of the Ring,...",0.000,0.687,0.550
5,6934,"Matrix Revolutions, The (2003)","Matrix Revolutions, The Action Adventure Sci-...",0.812,0.472,0.540
6,7153,"Lord of the Rings: The Return of the King, The...","Lord of the Rings: The Return of the King, The...",0.000,0.667,0.534
7,5952,"Lord of the Rings: The Two Towers, The (2002)","Lord of the Rings: The Two Towers, The Advent...",0.000,0.662,0.530
8,1196,Star Wars: Episode V - The Empire Strikes Back...,Star Wars: Episode V - The Empire Strikes Back...,0.000,0.658,0.526
9,1270,Back to the Future (1985),Back to the Future Adventure Comedy Sci-Fi 19...,0.212,0.594,0.518
